In [16]:
import subprocess
subprocess.run(["pip", "install", "einops", "nibabel", "-q"])

import os, json, time, gc, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
from scipy.ndimage import zoom, rotate, gaussian_filter, map_coordinates
from scipy.ndimage import label as scipy_label, binary_erosion, binary_dilation
import nibabel as nib
from einops import rearrange
warnings.filterwarnings('ignore')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Device: cuda
GPU: Tesla T4
VRAM: 15.6 GB


In [17]:
DATA_DIR = None
STAGE2_OLD_PATH = None

for root in ["/kaggle/input"]:
    for d in os.listdir(root):
        full = os.path.join(root, d)
        for dp, dn, fn in os.walk(full):
            if "dataset.json" in fn and os.path.exists(os.path.join(dp, "imagesTr")):
                DATA_DIR = dp
                print(f"✅ DATA_DIR = {dp}")
            for f in fn:
                if 'stage2' in f and f.endswith('.pth'):
                    candidate = os.path.join(dp, f)
                    if STAGE2_OLD_PATH is None or 'best' in f:
                        STAGE2_OLD_PATH = candidate
                        print(f"✅ STAGE2_OLD = {candidate}")

if not DATA_DIR: print("❌ Dataset not found!")
if not STAGE2_OLD_PATH: print("⚠️ No old Stage 2 checkpoint (will train from scratch)")

✅ STAGE2_OLD = /kaggle/input/models/arjungirinath/stage2v1/pytorch/default/1/stage2_best.pth
✅ DATA_DIR = /kaggle/input/datasets/rksrank1/pancreatic-cancer/Task07_Pancreas


In [19]:
CONFIG = {
    "data_dir": DATA_DIR,
    "cache_dir": "/kaggle/working/roi_cache_v2",
    "output_dir": "/kaggle/working",
    "patch_size": (64, 64, 64),
    "batch_size": 4,
    "accum_steps": 2,
    "lr": 5e-4,                # ← Lower LR for fine-tuning from old checkpoint
    "weight_decay": 1e-5,
    "epochs": 300,             # ← More epochs
    "val_split": 0.15,
    "num_workers": 2,
    "resume_from": STAGE2_OLD_PATH,   # ← Warm start from old Stage 2!
    "target_spacing": (1.5, 1.5, 2.5),
    "hu_window": (-125, 275),
    "roi_margin": 15,
    "fg_sample_rate": 0.85,    # ← Even more tumor-focused
    "use_cutmix": True,        # ← NEW: CutMix augmentation
    "cutmix_prob": 0.3,        # 30% chance per batch
    "ohem_ratio": 0.6,         # ← NEW: Keep hardest 60% of voxels
    "boundary_weight": 2.0,    # ← NEW: Extra weight on boundaries
}

os.makedirs(CONFIG["cache_dir"], exist_ok=True)
print("✅ Stage 2 v2 Config ready")

✅ Stage 2 v2 Config ready


In [27]:


def get_bbox(mask, margin=15):
    coords = np.argwhere(mask > 0)
    if len(coords) == 0: return None
    mins = np.maximum(coords.min(0) - margin, 0)
    maxs = np.minimum(coords.max(0) + 1 + margin, mask.shape)
    return tuple(slice(mn, mx) for mn, mx in zip(mins, maxs))

def preprocess_rois_gt(data_dir, cache_dir, target_spacing, hu_window, margin):
    # DIRECT FOLDER SCAN
    img_dir = os.path.join(data_dir, "imagesTr")
    lbl_dir = os.path.join(data_dir, "labelsTr")
    
    if not os.path.exists(img_dir):
        print(f"❌ Folder not found at: {img_dir}")
        return []

    # UPDATED: Look for .nii files specifically and ignore hidden '._' files
    image_filenames = sorted([f for f in os.listdir(img_dir) if f.endswith('.nii') and not f.startswith('._')])
    
    cases = []
    for f in image_filenames:
        img_path = os.path.join(img_dir, f)
        lbl_path = os.path.join(lbl_dir, f)
        if os.path.exists(lbl_path):
            cases.append((img_path, lbl_path))

    print(f"🚀 Preprocessing {len(cases)} ROIs from .nii files...")
    roi_files = []
    tumor_cases = 0

    for i, (img_path, lbl_path) in enumerate(tqdm(cases)):
        case_name = os.path.basename(img_path).replace(".nii", "")
        cache_path = os.path.join(cache_dir, f"{case_name}_roi.npz")
        
        if os.path.exists(cache_path):
            roi_files.append(cache_path)
            continue

        try:
            # Load Data
            ct = nib.load(img_path).get_fdata().astype(np.float32)
            label = nib.load(lbl_path).get_fdata().astype(np.int32)
            spacing = np.array(nib.load(img_path).header.get_zooms()[:3])
            
            # Resample to Target Spacing
            scale = spacing / np.array(target_spacing)
            ct = zoom(ct, scale, order=1)
            label = zoom(label, scale, order=0)
            
            # Clip Hounsfield Units and Normalize
            ct = np.clip(ct, hu_window[0], hu_window[1])
            fg = label > 0
            if fg.sum() > 100:
                ct = (ct - ct[fg].mean()) / (ct[fg].std() + 1e-8)
            else:
                ct = (ct - ct.mean()) / (ct.std() + 1e-8)

            # ROI Extraction based on the Pancreas mask
            bbox = get_bbox(label, margin)
            if bbox is None:
                ct_roi, label_roi = ct[:64, :64, :64], label[:64, :64, :64]
            else:
                ct_roi, label_roi = ct[bbox], label[bbox]

            # Track Tumor Presence (Class 2)
            has_tumor = (label_roi == 2).sum() > 0
            if has_tumor: tumor_cases += 1

            # Save as compressed .npz for your ViT-UNet training
            np.savez_compressed(cache_path,
                ct_roi=ct_roi.astype(np.float32),
                label_roi=label_roi.astype(np.int8),
                has_tumor=has_tumor
            )
            roi_files.append(cache_path)
            
            if (i+1) % 10 == 0:
                print(f"   [{i+1}/{len(cases)}] {case_name} ROI={ct_roi.shape} tumor={has_tumor}")
        
        except Exception as e:
            print(f"❌ Error processing {case_name}: {e}")

    print(f"✅ Done! {len(roi_files)} ROIs processed, {tumor_cases} with tumor segments.")
    return roi_files

# --- RUN EXECUTION ---
# Using the confirmed Kaggle path
DATA_PATH = "/kaggle/input/datasets/rksrank1/pancreatic-cancer/Task07_Pancreas"

roi_files = preprocess_rois_gt(
    DATA_PATH, 
    CONFIG["cache_dir"],
    CONFIG["target_spacing"], 
    CONFIG["hu_window"], 
    CONFIG["roi_margin"]
)

🚀 Preprocessing 281 ROIs from .nii files...


  0%|          | 0/281 [00:00<?, ?it/s]

   [10/281] pancreas_019 ROI=(102, 75, 60) tumor=True
   [20/281] pancreas_041 ROI=(124, 70, 64) tumor=True
   [30/281] pancreas_055 ROI=(115, 91, 67) tumor=True
   [40/281] pancreas_074 ROI=(96, 77, 59) tumor=True
   [50/281] pancreas_088 ROI=(131, 84, 72) tumor=True
   [60/281] pancreas_100 ROI=(121, 95, 60) tumor=True
   [70/281] pancreas_111 ROI=(129, 102, 68) tumor=True
   [80/281] pancreas_127 ROI=(138, 72, 63) tumor=True
   [90/281] pancreas_148 ROI=(109, 83, 56) tumor=True
   [100/281] pancreas_169 ROI=(90, 74, 65) tumor=True
   [110/281] pancreas_183 ROI=(90, 63, 53) tumor=True
   [120/281] pancreas_200 ROI=(125, 106, 73) tumor=True
   [130/281] pancreas_214 ROI=(121, 62, 58) tumor=True
   [140/281] pancreas_228 ROI=(98, 69, 68) tumor=True
   [150/281] pancreas_243 ROI=(119, 90, 64) tumor=True
   [160/281] pancreas_259 ROI=(97, 71, 55) tumor=True
   [170/281] pancreas_274 ROI=(124, 63, 72) tumor=True
   [180/281] pancreas_286 ROI=(101, 69, 66) tumor=True
   [190/281] pancreas_

In [28]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, dropout=0.0):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.InstanceNorm3d(out_ch, affine=True),
            nn.LeakyReLU(0.01, inplace=True),
            nn.Dropout3d(dropout) if dropout > 0 else nn.Identity(),
            nn.Conv3d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.InstanceNorm3d(out_ch, affine=True),
            nn.LeakyReLU(0.01, inplace=True),
        )
        self.residual = nn.Identity() if in_ch == out_ch else nn.Conv3d(in_ch, out_ch, 1, bias=False)
    def forward(self, x): return self.conv(x) + self.residual(x)

class DownBlock(nn.Module):
    def __init__(self, in_ch, out_ch, dropout=0.0):
        super().__init__()
        self.down = nn.Conv3d(in_ch, out_ch, 2, stride=2, bias=False)
        self.conv = ConvBlock(out_ch, out_ch, dropout)
    def forward(self, x): return self.conv(self.down(x))

class UpBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch, dropout=0.0):
        super().__init__()
        self.up = nn.ConvTranspose3d(in_ch, out_ch, 2, stride=2)
        self.conv = ConvBlock(out_ch + skip_ch, out_ch, dropout)
    def forward(self, x, skip):
        x = self.up(x)
        if x.shape != skip.shape:
            x = F.interpolate(x, size=skip.shape[2:], mode='trilinear', align_corners=False)
        return self.conv(torch.cat([x, skip], dim=1))

class PatchEmbedding3D(nn.Module):
    def __init__(self, in_ch, embed_dim, patch_size=2):
        super().__init__()
        self.proj = nn.Conv3d(in_ch, embed_dim, kernel_size=patch_size, stride=patch_size)
        self.norm = nn.LayerNorm(embed_dim)
    def forward(self, x):
        x = self.proj(x); B,C,D,H,W = x.shape
        x = rearrange(x, 'b c d h w -> b (d h w) c')
        return self.norm(x), (D,H,W)

class TransformerBlock(nn.Module):
    def __init__(self, dim, heads=6, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, int(dim*mlp_ratio)), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(int(dim*mlp_ratio), dim), nn.Dropout(dropout),
        )
    def forward(self, x):
        h = self.norm1(x); x = x + self.attn(h,h,h)[0]
        return x + self.mlp(self.norm2(x))

class ViTBottleneck(nn.Module):
    def __init__(self, in_ch, embed_dim=384, heads=6, depth=3, patch_size=2, dropout=0.1):
        super().__init__()
        self.patch_embed = PatchEmbedding3D(in_ch, embed_dim, patch_size)
        self.pos_embed = nn.Parameter(torch.randn(1, 27, embed_dim)*0.02)
        self.blocks = nn.Sequential(*[TransformerBlock(embed_dim, heads, dropout=dropout) for _ in range(depth)])
        self.norm = nn.LayerNorm(embed_dim)
        self.proj_back = nn.Linear(embed_dim, in_ch)
    def forward(self, x):
        B,C,D,H,W = x.shape
        tokens, (Dp,Hp,Wp) = self.patch_embed(x)
        N = tokens.shape[1]
        pos = F.interpolate(self.pos_embed.transpose(1,2), size=N, mode='linear', align_corners=False).transpose(1,2) if N!=27 else self.pos_embed
        tokens = self.blocks(tokens + pos)
        return rearrange(self.proj_back(self.norm(tokens)), 'b (d h w) c -> b c d h w', d=Dp, h=Hp, w=Wp)

class ViTUNet(nn.Module):
    def __init__(self, in_ch=1, num_classes=2, base=24, vit_dim=384, vit_depth=3, vit_heads=6, deep_sup=True):
        super().__init__()
        self.deep_sup = deep_sup
        self.enc1 = ConvBlock(in_ch, base); self.enc2 = DownBlock(base, base*2)
        self.enc3 = DownBlock(base*2, base*4, 0.1); self.enc4 = DownBlock(base*4, base*8, 0.1)
        self.down_bot = nn.Conv3d(base*8, base*8, 2, stride=2, bias=False)
        self.vit = ViTBottleneck(base*8, vit_dim, vit_heads, vit_depth, 2, 0.1)
        self.dec4 = UpBlock(base*8, base*8, base*8, 0.1); self.dec3 = UpBlock(base*8, base*4, base*4, 0.1)
        self.dec2 = UpBlock(base*4, base*2, base*2); self.dec1 = UpBlock(base*2, base, base)
        self.final = nn.Conv3d(base, num_classes, 1)
        if deep_sup:
            self.ds3 = nn.Conv3d(base*4, num_classes, 1)
            self.ds2 = nn.Conv3d(base*2, num_classes, 1)
        self._init_weights()
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv3d, nn.ConvTranspose3d)):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='leaky_relu')
            elif isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None: nn.init.constant_(m.bias, 0)
    def forward(self, x):
        s1=self.enc1(x); s2=self.enc2(s1); s3=self.enc3(s2); s4=self.enc4(s3)
        b = self.vit(self.down_bot(s4))
        d4=self.dec4(b,s4); d3=self.dec3(d4,s3); d2=self.dec2(d3,s2); d1=self.dec1(d2,s1)
        out = self.final(d1)
        if self.deep_sup and self.training: return out, self.ds3(d3), self.ds2(d2)
        return out

print(f"✅ ViTU-Net: {sum(p.numel() for p in ViTUNet(1,3,24).parameters())/1e6:.1f}M params")

✅ ViTU-Net: 13.7M params


In [29]:
# ==================== IMPROVED LOSS FUNCTIONS ====================

class SoftDiceLoss(nn.Module):
    def __init__(self, smooth=1e-5):
        super().__init__(); self.smooth = smooth
    def forward(self, pred, target):
        pred = F.softmax(pred, dim=1); nc = pred.shape[1]
        toh = F.one_hot(target.long(), nc).permute(0,4,1,2,3).float()
        scores = []
        for c in range(1, nc):
            p, t = pred[:,c].flatten(1), toh[:,c].flatten(1)
            scores.append((2*(p*t).sum(1)+self.smooth) / (p.sum(1)+t.sum(1)+self.smooth))
        return 1 - torch.stack(scores).mean()


class BoundaryAwareLoss(nn.Module):
    """
    Gives EXTRA weight to voxels near tumor/pancreas boundaries.
    The model struggles most at edges — this forces it to focus there.
    """
    def __init__(self, num_classes=3, boundary_dilation=2, boundary_weight=3.0):
        super().__init__()
        self.num_classes = num_classes
        self.dilation = boundary_dilation
        self.bw = boundary_weight

    def forward(self, pred, target):
        # Build per-voxel weight map
        weight_map = torch.ones_like(target, dtype=torch.float32)
        target_np = target.cpu().numpy()

        for b in range(target_np.shape[0]):
            for c in range(1, self.num_classes):
                mask = (target_np[b] == c).astype(np.uint8)
                if mask.sum() == 0:
                    continue
                eroded = binary_erosion(mask, iterations=self.dilation).astype(np.uint8)
                boundary = mask - eroded
                weight_map[b][torch.from_numpy(boundary > 0)] = self.bw

        weight_map = weight_map.to(pred.device)
        ce_loss = F.cross_entropy(pred, target.long(), reduction='none')
        return (ce_loss * weight_map).mean()


class OHEMDiceCELoss(nn.Module):
    """
    Combined loss:
    1. Dice Loss (global shape matching)
    2. Boundary-Aware CE (focus on edges)
    3. Online Hard Example Mining (focus on mistakes)
    """
    def __init__(self, class_weights=None, ohem_ratio=0.6, boundary_weight=3.0, num_classes=3):
        super().__init__()
        self.dice = SoftDiceLoss()
        self.boundary = BoundaryAwareLoss(num_classes, 2, boundary_weight)
        self.ohem_ratio = ohem_ratio
        w = torch.tensor(class_weights).float().cuda() if class_weights else None
        self.ce = nn.CrossEntropyLoss(weight=w, reduction='none')

    def forward(self, pred, target):
        # 1. Dice loss (always full)
        dice_loss = self.dice(pred, target)

        # 2. OHEM CE loss (keep hardest voxels)
        ce_per_voxel = self.ce(pred, target.long())
        k = max(1, int(ce_per_voxel.numel() * self.ohem_ratio))
        top_losses, _ = torch.topk(ce_per_voxel.flatten(), k)
        ohem_ce = top_losses.mean()

        # 3. Boundary loss
        boundary_loss = self.boundary(pred, target)

        # Combine: 40% dice + 30% OHEM CE + 30% boundary
        return 0.4 * dice_loss + 0.3 * ohem_ce + 0.3 * boundary_loss


class DeepSupLoss(nn.Module):
    def __init__(self, base_loss, weights=[1.0, 0.5, 0.25]):
        super().__init__(); self.base = base_loss; self.weights = weights
    def forward(self, outputs, target):
        if not isinstance(outputs, tuple): return self.base(outputs, target)
        total = 0.0
        for pred, w in zip(outputs, self.weights):
            if pred.shape[2:] != target.shape[1:]:
                t = F.interpolate(target.unsqueeze(1).float(), size=pred.shape[2:], mode='nearest').squeeze(1).long()
            else: t = target
            total += w * self.base(pred, t)
        return total


def compute_dice(pred, target, num_classes=3):
    if pred.ndim == 5: pred = pred.argmax(dim=1)
    scores = {}
    for c in range(1, num_classes):
        p = (pred==c).float().flatten(1); t = (target==c).float().flatten(1)
        inter = (p*t).sum(1); union = p.sum(1)+t.sum(1); mask = union > 0
        if mask.sum() > 0: scores[f"c{c}"] = (2*inter[mask]/(union[mask]+1e-8)).mean().item()
        else: scores[f"c{c}"] = float('nan')
    valid = [v for v in scores.values() if v==v]
    scores["mean"] = sum(valid)/len(valid) if valid else 0.0
    return scores


class PolyLRScheduler:
    def __init__(self, opt, max_ep, power=0.9):
        self.opt=opt; self.max_ep=max_ep; self.power=power
        self.base_lrs=[pg['lr'] for pg in opt.param_groups]
    def step(self, ep):
        f = (1-ep/self.max_ep)**self.power
        for pg, blr in zip(self.opt.param_groups, self.base_lrs): pg['lr']=blr*f
    def state_dict(self): return {'base_lrs':self.base_lrs}
    def load_state_dict(self, s): self.base_lrs=s['base_lrs']


print("✅ Improved losses ready:")
print("  - SoftDiceLoss (global shape)")
print("  - BoundaryAwareLoss (edge focus)")
print("  - OHEMDiceCELoss (hard example mining)")
print("  - DeepSupLoss (multi-scale)")

✅ Improved losses ready:
  - SoftDiceLoss (global shape)
  - BoundaryAwareLoss (edge focus)
  - OHEMDiceCELoss (hard example mining)
  - DeepSupLoss (multi-scale)


In [30]:
# ==================== AUGMENTATION FUNCTIONS ====================

def elastic_deform(image, label, alpha=80, sigma=8):
    shape = image.shape
    dx = gaussian_filter(np.random.randn(*shape)*alpha, sigma, mode='constant')
    dy = gaussian_filter(np.random.randn(*shape)*alpha, sigma, mode='constant')
    dz = gaussian_filter(np.random.randn(*shape)*alpha, sigma, mode='constant')
    z,y,x = np.meshgrid(np.arange(shape[0]),np.arange(shape[1]),np.arange(shape[2]),indexing='ij')
    coords = [np.clip(z+dz,0,shape[0]-1),np.clip(y+dy,0,shape[1]-1),np.clip(x+dx,0,shape[2]-1)]
    return map_coordinates(image,coords,order=1,mode='reflect').astype(np.float32), \
           map_coordinates(label.astype(float),coords,order=0,mode='reflect').astype(np.int32)

def gamma_aug(image, gamma_range=(0.7,1.5)):
    mn=image.min(); s=image-mn+1e-8; mx=s.max()+1e-8
    g=np.random.uniform(*gamma_range); out=np.power(s/mx,g)*mx+mn
    if np.random.random()<0.5: out=-out+image.mean()*2
    return out.astype(np.float32)

def cutmix_3d(img1, lbl1, img2, lbl2, alpha=0.4):
    """
    CutMix: paste a random cube from sample2 into sample1.
    Creates harder training examples and improves generalization.
    """
    lam = np.random.beta(alpha, alpha)
    D, H, W = img1.shape[-3:]

    cut_d = int(D * (1 - lam) ** (1/3))
    cut_h = int(H * (1 - lam) ** (1/3))
    cut_w = int(W * (1 - lam) ** (1/3))

    d = np.random.randint(0, max(1, D - cut_d))
    h = np.random.randint(0, max(1, H - cut_h))
    w = np.random.randint(0, max(1, W - cut_w))

    img1[..., d:d+cut_d, h:h+cut_h, w:w+cut_w] = img2[..., d:d+cut_d, h:h+cut_h, w:w+cut_w]
    lbl1[..., d:d+cut_d, h:h+cut_h, w:w+cut_w] = lbl2[..., d:d+cut_d, h:h+cut_h, w:w+cut_w]
    return img1, lbl1


class ROIDatasetV2(Dataset):
    """Stage 2 dataset with CutMix support"""
    def __init__(self, files, patch_size=(64,64,64), augment=True, fg_rate=0.85):
        self.files = files; self.ps = patch_size
        self.augment = augment; self.fg_rate = fg_rate
        print(f"ROI Dataset v2: {len(files)} ROIs, patch={patch_size}")

    def __len__(self):
        return len(self.files) * 8  # More patches per epoch

    def __getitem__(self, idx):
        data = np.load(self.files[idx % len(self.files)])
        ct, label = data['ct_roi'], data['label_roi'].astype(np.int32)

        img, lbl = self._patch(ct, label)
        if self.augment: img, lbl = self._aug(img, lbl)
        return torch.from_numpy(img).unsqueeze(0).float(), torch.from_numpy(lbl).long()

    def _patch(self, image, label):
        D,H,W = image.shape; pd,ph,pw = self.ps
        if np.random.random() < self.fg_rate:
            fg = np.argwhere(label == 2)
            if len(fg) < 5: fg = np.argwhere(label > 0)
            if len(fg) > 0:
                c = fg[np.random.randint(len(fg))] + np.random.randint(-pd//4, pd//4+1, size=3)
                d = np.clip(c[0]-pd//2, 0, max(0,D-pd))
                h = np.clip(c[1]-ph//2, 0, max(0,H-ph))
                w = np.clip(c[2]-pw//2, 0, max(0,W-pw))
            else:
                d,h,w = [np.random.randint(0,max(1,s-p+1)) for s,p in zip(image.shape,self.ps)]
        else:
            d,h,w = [np.random.randint(0,max(1,s-p+1)) for s,p in zip(image.shape,self.ps)]
        ip = image[d:d+pd, h:h+ph, w:w+pw]
        lp = label[d:d+pd, h:h+ph, w:w+pw]
        if ip.shape != tuple(self.ps):
            pi,pl = np.zeros(self.ps,np.float32), np.zeros(self.ps,np.int32)
            s=ip.shape; pi[:s[0],:s[1],:s[2]]=ip; pl[:s[0],:s[1],:s[2]]=lp
            ip,lp = pi,pl
        return ip, lp

    def _aug(self, img, lbl):
        for ax in range(3):
            if np.random.random()<0.5: img=np.flip(img,ax).copy(); lbl=np.flip(lbl,ax).copy()
        if np.random.random()<0.3:
            ang=np.random.uniform(-15,15); axes=[(0,1),(0,2),(1,2)][np.random.randint(3)]
            img=rotate(img,ang,axes=axes,reshape=False,order=1,mode='reflect')
            lbl=rotate(lbl.astype(float),ang,axes=axes,reshape=False,order=0,mode='reflect').astype(np.int32)
        if np.random.random()<0.25:
            img,lbl=elastic_deform(img,lbl)
        if np.random.random()<0.3:
            img=gamma_aug(img)
        if np.random.random()<0.2:
            img=img+np.random.normal(0,0.02,img.shape).astype(np.float32)
        if np.random.random()<0.3:
            img=img*np.random.uniform(0.9,1.1)+np.random.uniform(-0.1,0.1)
        if np.random.random()<0.15:
            img=gaussian_filter(img,sigma=np.random.uniform(0.5,1.0))
        return img.astype(np.float32), lbl.astype(np.int32)

td = ROIDatasetV2(roi_files[:2], CONFIG["patch_size"])
i,l = td[0]
print(f"✅ Test: img={i.shape}, lbl={l.shape}, unique={l.unique().tolist()}")
del td

ROI Dataset v2: 2 ROIs, patch=(64, 64, 64)
✅ Test: img=torch.Size([1, 64, 64, 64]), lbl=torch.Size([64, 64, 64]), unique=[0, 1, 2]


In [31]:
def train_stage2_v2():
    np.random.seed(42)
    idx = np.random.permutation(len(roi_files))
    vs = int(len(roi_files) * CONFIG["val_split"])
    vf = [roi_files[i] for i in idx[:vs]]
    tf = [roi_files[i] for i in idx[vs:]]

    tds = ROIDatasetV2(tf, CONFIG["patch_size"], True, CONFIG["fg_sample_rate"])
    vds = ROIDatasetV2(vf, CONFIG["patch_size"], False, 0.5)
    tl = DataLoader(tds, CONFIG["batch_size"], True, num_workers=CONFIG["num_workers"], pin_memory=True, drop_last=True)
    vl = DataLoader(vds, CONFIG["batch_size"], False, num_workers=CONFIG["num_workers"], pin_memory=True)

    # Model (3 classes)
    model = ViTUNet(1, 3, 24, 384, 3, 6).to(device)

    # NEW: Improved loss with boundary awareness + OHEM
    base_loss = OHEMDiceCELoss(
        class_weights=[0.15, 0.25, 0.60],
        ohem_ratio=CONFIG["ohem_ratio"],
        boundary_weight=CONFIG["boundary_weight"],
        num_classes=3
    )
    criterion = DeepSupLoss(base_loss)

    optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
    scheduler = PolyLRScheduler(optimizer, CONFIG["epochs"])
    scaler = GradScaler()

    start_epoch, best_dice, best_tumor = 0, 0.0, 0.0
    hist = {"loss":[], "dice":[], "panc":[], "tumor":[]}

    # Warm start from old Stage 2
    if CONFIG["resume_from"] and os.path.exists(CONFIG["resume_from"]):
        print(f"🔄 Loading weights from {CONFIG['resume_from']}...")
        try:
            ck = torch.load(CONFIG["resume_from"], map_location=device)
            model.load_state_dict(ck['model_state_dict'])
            # DON'T load optimizer/scheduler — fresh start with new loss
            old_tumor = ck.get('best_tumor', ck.get('best_dice', '?'))
            print(f"✅ Loaded model weights. Old best tumor: {old_tumor}")
            print(f"   Starting fresh optimizer (new loss function)")
        except Exception as e:
            print(f"⚠️ Could not load checkpoint: {e}. Training from scratch.")

    print(f"\n{'='*60}")
    print(f"  STAGE 2 v2: TUMOR SEGMENTATION (IMPROVED LOSSES)")
    print(f"  New: Boundary Loss + OHEM + CutMix")
    print(f"  {start_epoch} → {CONFIG['epochs']} epochs")
    print(f"{'='*60}\n")

    for epoch in range(start_epoch, CONFIG["epochs"]):
        t0 = time.time(); model.train(); rl = 0; optimizer.zero_grad()

        for i, (imgs, lbls) in enumerate(tl):
            imgs, lbls = imgs.to(device), lbls.to(device)

            # CutMix augmentation (batch-level)
            if CONFIG["use_cutmix"] and np.random.random() < CONFIG["cutmix_prob"]:
                # Shuffle batch and mix
                perm = torch.randperm(imgs.shape[0])
                imgs, lbls = cutmix_3d(
                    imgs.clone(), lbls.clone(),
                    imgs[perm], lbls[perm]
                )

            with autocast():
                loss = criterion(model(imgs), lbls) / CONFIG["accum_steps"]
            scaler.scale(loss).backward()
            if (i+1) % CONFIG["accum_steps"] == 0:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer); scaler.update(); optimizer.zero_grad()
            rl += loss.item() * CONFIG["accum_steps"]

        scheduler.step(epoch)
        tl_loss = rl / max(len(tl), 1)

        # Validate
        do_val = (epoch % 5 == 0) or (epoch >= CONFIG["epochs"] - 100)
        if do_val:
            model.eval()
            vd,vp,vt,cnt = 0,0,0,0
            with torch.no_grad():
                for imgs, lbls in vl:
                    imgs, lbls = imgs.to(device), lbls.to(device)
                    with autocast():
                        pred = model(imgs)
                        if isinstance(pred,tuple): pred=pred[0]
                    d = compute_dice(pred, lbls, 3)
                    vd += d["mean"]
                    vp += d.get("c1",0) if d.get("c1",0)==d.get("c1",0) else 0
                    vt += d.get("c2",0) if d.get("c2",0)==d.get("c2",0) else 0
                    cnt += 1
            val_dice = vd/max(cnt,1); val_panc = vp/max(cnt,1); val_tumor = vt/max(cnt,1)
        else:
            val_dice = hist["dice"][-1] if hist["dice"] else 0
            val_panc = hist["panc"][-1] if hist["panc"] else 0
            val_tumor = hist["tumor"][-1] if hist["tumor"] else 0

        hist["loss"].append(tl_loss); hist["dice"].append(val_dice)
        hist["panc"].append(val_panc); hist["tumor"].append(val_tumor)
        el = time.time()-t0; lr = optimizer.param_groups[0]['lr']

        star = ""
        if val_tumor > best_tumor:
            best_tumor = val_tumor; best_dice = val_dice
            torch.save({'epoch':epoch, 'model_state_dict':model.state_dict(),
                'optimizer_state_dict':optimizer.state_dict(), 'scaler_state_dict':scaler.state_dict(),
                'best_dice':best_dice, 'best_tumor':best_tumor, 'history':hist},
                os.path.join(CONFIG["output_dir"], "stage2_v2_best.pth"))
            star = " ★"

        # Save latest every epoch (crash protection)
        torch.save({'epoch':epoch, 'model_state_dict':model.state_dict(),
            'optimizer_state_dict':optimizer.state_dict(), 'scaler_state_dict':scaler.state_dict(),
            'best_dice':best_dice, 'best_tumor':best_tumor, 'history':hist},
            os.path.join(CONFIG["output_dir"], "stage2_v2_latest.pth"))

        if do_val or star:
            print(f"E{epoch:03d} ({el:.0f}s) Loss:{tl_loss:.4f} Mean:{val_dice:.4f} "
                  f"Panc:{val_panc:.4f} Tumor:{val_tumor:.4f} LR:{lr:.6f}{star}")

        if (epoch+1) % 50 == 0:
            torch.save({'epoch':epoch, 'model_state_dict':model.state_dict(),
                'optimizer_state_dict':optimizer.state_dict(), 'scaler_state_dict':scaler.state_dict(),
                'best_dice':best_dice, 'best_tumor':best_tumor, 'history':hist},
                os.path.join(CONFIG["output_dir"], f"stage2_v2_ep{epoch}.pth"))
            print(f"  💾 stage2_v2_ep{epoch}.pth")

    print(f"\n✅ Done! Best Tumor: {best_tumor:.4f} | Best Mean: {best_dice:.4f}")
    return hist

history_v2 = train_stage2_v2()

ROI Dataset v2: 239 ROIs, patch=(64, 64, 64)
ROI Dataset v2: 42 ROIs, patch=(64, 64, 64)
🔄 Loading weights from /kaggle/input/models/arjungirinath/stage2v1/pytorch/default/1/stage2_best.pth...
✅ Loaded model weights. Old best tumor: 0.43951025794422816
   Starting fresh optimizer (new loss function)

  STAGE 2 v2: TUMOR SEGMENTATION (IMPROVED LOSSES)
  New: Boundary Loss + OHEM + CutMix
  0 → 300 epochs

E000 (117s) Loss:0.2038 Mean:0.8099 Panc:0.8876 Tumor:0.7322 LR:0.000500 ★
E005 (119s) Loss:0.1837 Mean:0.8068 Panc:0.8873 Tumor:0.7263 LR:0.000492
E010 (118s) Loss:0.1854 Mean:0.8076 Panc:0.8803 Tumor:0.7349 LR:0.000485 ★
E015 (119s) Loss:0.1714 Mean:0.8013 Panc:0.8804 Tumor:0.7221 LR:0.000477
E020 (121s) Loss:0.1727 Mean:0.8033 Panc:0.8790 Tumor:0.7276 LR:0.000470
E025 (122s) Loss:0.1681 Mean:0.7866 Panc:0.8764 Tumor:0.6968 LR:0.000462
E030 (118s) Loss:0.1622 Mean:0.7732 Panc:0.8651 Tumor:0.6813 LR:0.000455
E035 (119s) Loss:0.1652 Mean:0.7822 Panc:0.8751 Tumor:0.6894 LR:0.000447
E040